# Tutorial 4 — OMOP Harmonization with the Rhino DHE

This notebook maps the three prepared datasets to OMOP CDM target tables:
- `patients` (prepared) → **OMOP Person**
- `encounters` (prepared) → **OMOP Visit Occurrence**
- `procedures` (prepared) → **OMOP Procedure Occurrence**

**Can I do this in the UI instead?**
Yes — all harmonization steps (creating mappings, approving semantic terms,
running harmonization) can be done via the FCP Dashboard at
https://dashboard.rhinohealth.com/login under Projects → Data Mappings.
The notebook automates those same steps.

**What standard are we using?**
This tutorial uses **OMOP CDM**. The Rhino DHE also supports **FHIR** and
**custom target models** — only the `target_data_model_type` parameter and
the target schema UIDs change. The full workflow structure is identical.

---
**Prerequisites:**
- Tutorial 3 complete — all six prepared UIDs required below
- OMOP target schemas pre-defined in the FCP (person, visit_occurrence, procedure_occurrence)
- Vocabulary UIDs for Gender, Race, Ethnicity, Visit Type, CPT4 (retrieved in Step 2)

**Inputs:** Prepared datasets and schemas from Tutorial 3

**Outputs:** Three OMOP-conformant datasets on the Rhino client, plus the
reusable semantic and syntactic mapping objects stored in the FCP

## Configuration

In [ ]:
import os
import time
import rhino_health as rh
from getpass import getpass
from rhino_health import ApiEnvironment

# From Tutorial 3 — paste your values here
PROJECT_UID = "<YOUR-PROJECT-UID>"  # REPLACE with your project UID"

PREPARED_PATIENTS_UID          = '<YOUR-PREPARED-PATIENTS-UID>'             # REPLACE with your Prepared Patients dataset UID
PREPARED_ENCOUNTERS_UID        = '<YOUR-PREPARED-ENCOUNTERS-UID>'           # REPLACE with your Prepared Encounters dataset UID
PREPARED_PROCEDURES_UID        = '<YOUR-PREPARED-PROCEDURES-UID>'           # REPLACE with your Prepared Procedures dataset UID
PREPARED_PATIENTS_SCHEMA_UID   = '<YOUR-PREPARED-PATIENTS-SCHEMA-UID>'      # REPLACE with your Prepared Patients schema UID
PREPARED_ENCOUNTERS_SCHEMA_UID = '<YOUR-PREPARED-ENCOUNTERS-SCHEMA-UID>'    # REPLACE with your Prepared Encounters schema UID
PREPARED_PROCEDURES_SCHEMA_UID = '<YOUR-PREPARED-PROCEDURES-SCHEMA-UID>'    # REPLACE with your Prepared Procedures schema UID

if any(v.startswith("<") for v in [PROJECT_UID,
    PREPARED_PATIENTS_UID, PREPARED_ENCOUNTERS_UID, PREPARED_PROCEDURES_UID,
    PREPARED_PATIENTS_SCHEMA_UID, PREPARED_ENCOUNTERS_SCHEMA_UID, PREPARED_PROCEDURES_SCHEMA_UID]):
    raise ValueError("Please fill in all UIDs from Tutorial 3 before continuing.")

print("Configuration loaded.")

## Step 1 — Authenticate

In [ ]:
my_username = "<YOUR-USERNAME>"  # Note that to fully run this notebook, your FCP account MUST have the site-level role permission to "Manage Data Mappings"
session = rh.login(username=my_username, password=getpass(), rhino_api_url=ApiEnvironment.PROD_AWS_URL) # e.g., PROD_AWS_URL, STAGING_AWS_URL, DEV2_AWS_URL, SOLUTIONS_GCP_URL
print(f"Logged in as: {my_username}")

project   = session.project.get_projects(project_uids=[PROJECT_UID])[0]
workgroup = session.project.get_collaborating_workgroups(PROJECT_UID)[0]
WORKGROUP_UID = workgroup.uid

print(f"Project \t({project.uid}): \t{project.name}")
print(f"Workgroup \t({workgroup.uid}): \t{workgroup.name}")

## Step 2 — Create OMOP Target Schemas

The three OMOP target schemas are defined as CSV files in the `schemas/` subfolder alongside this notebook. Each file describes the columns of one OMOP CDM table using the same transposed format used in Tutorial 1 (rows = attributes like `Variable Name`, `Type`; columns = fields).

The cell below registers them on the FCP via the SDK. If schemas with these names already exist in the project (e.g. from a previous run), they are reused automatically — nothing is overwritten.

**Why CSV files instead of inline code?**
Schema definitions are stable reference data, not runtime logic. Keeping them as CSV files makes them easy to inspect, diff in git, and reuse across projects without touching the notebook.

| File | Target Table | Source Dataset |
|---|---|---|
| `schemas/omop_person.csv` | `person` | Patients (prepared) |
| `schemas/omop_visit_occurrence.csv` | `visit_occurrence` | Encounters (prepared) |
| `schemas/omop_procedure_occurrence.csv` | `procedure_occurrence` | Procedures (prepared) |

In [ ]:
from rhino_health.lib.endpoints.data_schema.data_schema_dataclass import DataSchemaCreateInput

# Extract workgroup UID
_wg_uid = WORKGROUP_UID.uid if hasattr(WORKGROUP_UID, "uid") else str(WORKGROUP_UID)
WORKGROUP_UID = _wg_uid

# add schemas dir at the same level as this notebook (assumes you're running the notebook from its current location)
_SCHEMA_DIR = os.path.join(os.getcwd(), "schemas")

def _get_or_create_omop_schema(name, description, csv_filename):
    """Return UID of an existing schema by name, or create it from the CSV file."""
    existing = session.data_schema.get_data_schema_by_name(name, project_uid=PROJECT_UID)
    if existing:
        print(f"  '{name}' already exists — reusing {existing.uid}")
        return existing.uid
    schema = session.data_schema.create_data_schema(
        DataSchemaCreateInput(
            name=name,
            description=description,
            primary_workgroup_uid=_wg_uid,
            project_uid=PROJECT_UID,
            file_path=os.path.join(_SCHEMA_DIR, csv_filename),
        ),
        return_existing=False,
    )
    print(f"  '{name}' created — {schema.uid}")
    return schema.uid

print(f"Using workgroup UID: {_wg_uid}")
print("Creating OMOP target schemas...")
OMOP_PERSON_SCHEMA_UID = _get_or_create_omop_schema(
    name="OMOP Person",
    description="OMOP CDM Person table — target schema for patient demographic harmonization",
    csv_filename="omop_person.csv",
)
OMOP_VISIT_OCCURRENCE_SCHEMA_UID = _get_or_create_omop_schema(
    name="OMOP Visit Occurrence",
    description="OMOP CDM Visit Occurrence table — target schema for encounter harmonization",
    csv_filename="omop_visit_occurrence.csv",
)
OMOP_PROCEDURE_OCCURRENCE_SCHEMA_UID = _get_or_create_omop_schema(
    name="OMOP Procedure Occurrence",
    description="OMOP CDM Procedure Occurrence table — target schema for procedure harmonization",
    csv_filename="omop_procedure_occurrence.csv",
)

print(f"\nOMOP_PERSON_SCHEMA_UID               = '{OMOP_PERSON_SCHEMA_UID}'")
print(f"OMOP_VISIT_OCCURRENCE_SCHEMA_UID     = '{OMOP_VISIT_OCCURRENCE_SCHEMA_UID}'")
print(f"OMOP_PROCEDURE_OCCURRENCE_SCHEMA_UID = '{OMOP_PROCEDURE_OCCURRENCE_SCHEMA_UID}'")

## Step 3 — Create Syntactic Mapping and Semantic Mapping Objects

- The **Syntactic Mapping** defines the structural transformation (which source columns map to which target fields)
- The **Semantic Mappings** define value-level vocabulary translations (e.g. `"Male"` → OMOP concept `8507`) and are created in **Needs Review** status, (pending approval)

Before creating the encounter mappings, we also create a **Custom Vocabulary** called `"Custom Encounter Type Codes"` to represent the source encounter type values used in the `TypeOfService` column. This custom vocabulary is then referenced in the semantic mapping for the encounters → OMOP Visit Occurrence table pair.

**Note on target_data_model_type:**
This tutorial uses `"omop"`. Change to `"fhir"` for FHIR targets or
`"custom"` for any non-standard target schema you have defined.

### 3a — Create Syntactic Mapping

A **Syntactic Mapping** defines how source data is structurally transformed into a target data model. Creating one involves three components:

1. **`global_configuration`** — sets up the mapping context:
   - `data_sources_to_data_schemas` — maps a snake_case source name (e.g. `"patients"`) to its prepared source schema UID. This tells the mapping engine which schema describes each input dataset.
   - `vocabulary_mapping` — lists any vocabularies used for semantic lookups (e.g. OMOP concept IDs). Left empty here and populated in Step 2b when semantic mappings are created.

2. **`target_data_model_type`** — the target standard: `OMOP`, `FHIR`, or `CUSTOM`.

3. **`table_configurations`** — one entry per target OMOP table, each with:
   - `table_name` — the OMOP target table (e.g. `"person"`, `"visit_occurrence"`, `"procedure_occurrence"`)
   - `field_configurations` — the column-level transformation rules. Left empty here; auto-generated in a later step.

The result is a single syntactic mapping object called `"Data to OMOP"` that covers all three source datasets in one run.

In [ ]:
from rhino_health.lib.endpoints.syntactic_mapping.syntactic_mapping_dataclass import (
    SyntacticMappingCreateInput, SyntacticMappingDataModel,
    SyntacticMappingConfig, GlobalConfiguration, TableConfiguration
)

_SM_NAME = "Map All Prepared Data to OMOP"

# Delete ALL existing syntactic mappings with this name (including locked versions
# from prior runs). A mapping becomes locked once used in a code run, so we must
# start fresh each time this cell executes.
_existing_sms = session.syntactic_mapping.search_for_syntactic_mappings_by_name(
    _SM_NAME, project_uid=PROJECT_UID
)
for _sm in _existing_sms:
    try:
        session.syntactic_mapping.remove_syntactic_mapping(_sm.uid)
        print(f"Deleted existing syntactic mapping: {_sm.uid}")
    except Exception as _del_err:
        print(f"  Warning: could not delete {_sm.uid}: {_del_err}")

# output_table_names_to_data_schemas tells the platform how many output directories
# to create on the client (/output/0/, /output/1/, /output/2/) and which OMOP schema
# each output table conforms to. Without this, the container crashes with:
#   OSError: Cannot save file into a non-existent directory: '/output/0'
syntactic_mapping = session.syntactic_mapping.create_syntactic_mapping(
    SyntacticMappingCreateInput(
        name=_SM_NAME,
        primary_workgroup_uid=WORKGROUP_UID,
        project_uid=PROJECT_UID,
        target_data_model_type=SyntacticMappingDataModel.OMOP,
        mapping_config=SyntacticMappingConfig(
            global_configuration=GlobalConfiguration(
                data_sources_to_data_schemas=[
                    {"patients":   PREPARED_PATIENTS_SCHEMA_UID},
                    {"encounters": PREPARED_ENCOUNTERS_SCHEMA_UID},
                    {"procedures": PREPARED_PROCEDURES_SCHEMA_UID},
                ],
                vocabulary_mapping=[],
                output_table_names_to_data_schemas=[
                    {"person":               OMOP_PERSON_SCHEMA_UID},
                    {"visit_occurrence":     OMOP_VISIT_OCCURRENCE_SCHEMA_UID},
                    {"procedure_occurrence": OMOP_PROCEDURE_OCCURRENCE_SCHEMA_UID},
                ],
            ),
            table_configurations=[
                TableConfiguration(table_name="person",               field_configurations=[]),
                TableConfiguration(table_name="visit_occurrence",     field_configurations=[]),
                TableConfiguration(table_name="procedure_occurrence",  field_configurations=[]),
            ],
        ),
    ),
    return_existing=False,
)
SYNTACTIC_MAPPING_UID = syntactic_mapping.uid
print(f"Syntactic mapping created: {syntactic_mapping.name}")
print(f"  UID: {SYNTACTIC_MAPPING_UID}")
print(f"  Note: field configurations are populated in Step 4.")

### 3b — Create Semantic Mappings

A **Semantic Mapping** translates individual source values to target vocabulary concepts (e.g., `"Asian"` → OMOP concept `8515`). Each mapping links an **input vocabulary** (custom — the unique source values for one column) to an **output vocabulary** (the OMOP standard vocabulary on the platform).

This cell does three things in sequence:
1. **Discovers** the OMOP standard vocabulary already on the platform
2. **Creates custom input vocabularies** — one per source column, populated with the known source values
3. **Creates semantic mappings** — pairs each input vocabulary with the OMOP output vocabulary, then waits for the AI term-matching to finish

Five semantic mappings are created (Gender, Race, Ethnicity, Visit Type, CPT4 codes) covering all three source datasets.

In [ ]:
from rhino_health.lib.endpoints.semantic_mapping.semantic_mapping_dataclass import (
    SemanticMappingCreateInput, DatasetColumn, VocabularyInput,
)

# ── 1. Discover the OMOP standard vocabulary via raw API ──────────────────────
# The system OMOP vocabulary has null workgroup/project, which trips the SDK's
# Pydantic model — use the raw response to extract the UID directly.
print("Discovering OMOP vocabulary (raw API)...")
raw = session.get("/vocabularies", params={"name": "OMOP"})
vocab_data = raw.raw_response.json().get("data", [])
omop_entries = [v for v in vocab_data if v.get("name") == "OMOP" and v.get("type") == "standard"]
if not omop_entries:
    print("All vocabularies found:")
    for v in vocab_data:
        print(f"  [{v.get('type')}] {v.get('name')}  uid={v.get('uid')}")
    raise RuntimeError("No standard 'OMOP' vocabulary found — see list above.")

OMOP_VOCABULARY_UID = omop_entries[0]["uid"]
print(f"  OMOP vocabulary uid: {OMOP_VOCABULARY_UID}")

# ── 2. Helpers ────────────────────────────────────────────────────────────────
def _create_source_vocab(name, terms):
    """Create (or reuse) a custom vocabulary of source terms. Returns UID."""
    vocab = session.vocabulary.create_vocabulary(
        VocabularyInput(
            name=name,
            primary_workgroup_uid=WORKGROUP_UID,
            project_uid=PROJECT_UID,
            terms=[{"term_identifier": t, "term_display_name": t} for t in terms],
        ),
        return_existing=True,
    )
    print(f"  vocab '{name}': {vocab.uid}")
    return vocab.uid

def _create_semantic_mapping(name, input_vocab_uid, categories, dataset_uid=None, field_name=None):
    """Create (or reuse) a semantic mapping. Returns the SemanticMapping object.

    dataset_uid / field_name: when provided, the platform scans the actual dataset
    column to seed source term suggestions alongside the vocabulary terms.
    NOTE: passing source_dataset_columns=[] or None causes a backend 500 ("list index
    out of range") — always provide a real dataset column or omit both arguments.
    """
    source_cols = (
        [DatasetColumn(dataset_uid=dataset_uid, field_name=field_name)]
        if dataset_uid and field_name
        else []
    )
    sm = session.semantic_mapping.create_semantic_mapping(
        SemanticMappingCreateInput(
            name=name,
            primary_workgroup_uid=WORKGROUP_UID,
            project_uid=PROJECT_UID,
            input_vocabulary_uid=input_vocab_uid,
            output_vocabulary_uid=OMOP_VOCABULARY_UID,
            output_vocabulary_categories=categories,
            source_dataset_columns=source_cols,
        ),
        return_existing=True,
    )
    print(f"  mapping '{name}': {sm.uid}  [{sm.processing_status.value}]")
    return sm

# ── 3. Create source vocabularies ────────────────────────────────────────────
print("\nCreating source vocabularies...")
_gender_vocab_uid    = _create_source_vocab("Source: Gender Values",     ["Male", "Female", "Other"])
_race_vocab_uid      = _create_source_vocab("Source: Race Values",       ["Asian", "Black", "White", "Other"])
_ethnicity_vocab_uid = _create_source_vocab("Source: Ethnicity Values",  ["Hispanic", "Non-Hispanic"])
_visit_vocab_uid     = _create_source_vocab("Source: Visit Type Values", ["Outpatient", "Inpatient", "Emergency"])
_cpt_vocab_uid       = _create_source_vocab("Source: CPT4 Codes",        ["45378", "44950", "99203", "99212", "99213", "99285"])

# ── 4. Create semantic mappings ───────────────────────────────────────────────
# NOTE on CPT: The prepared procedures dataset stores ProcedureCode as floats
# (e.g. 99285.0) because pandas reads numeric columns that way. When the platform
# scans the column via source_dataset_columns, it creates entries keyed by float
# strings ("99285.0"). The source vocabulary terms produce entries keyed by int
# strings ("99285"). Both entry types exist after processing.
#
# At harmonization time, the syntactic mapping applies _normalize_cpt
# (99285.0 → "99285") BEFORE the semantic lookup, so the lookup key is int format.
# Step 4 therefore approves BOTH formats so all entries are approved and the
# overall mapping status reaches "Approved".
#
# source_dataset_columns must point to a real column — passing [] or None causes
# a backend 500 ("list index out of range") regardless of SDK version.
print("\nCreating semantic mappings...")
_gender_sm    = _create_semantic_mapping("Normalize Gender [OMOP]",     _gender_vocab_uid,    ["Gender"],     PREPARED_PATIENTS_UID,    "Gender")
_race_sm      = _create_semantic_mapping("Normalize Race [OMOP]",       _race_vocab_uid,      ["Race"],       PREPARED_PATIENTS_UID,    "Race")
_ethnicity_sm = _create_semantic_mapping("Normalize Ethnicity [OMOP]",  _ethnicity_vocab_uid, ["Ethnicity"],  PREPARED_PATIENTS_UID,    "Ethnicity")
_visit_sm     = _create_semantic_mapping("Normalize Visit Type [OMOP]", _visit_vocab_uid,     ["Visit"],      PREPARED_ENCOUNTERS_UID,  "TypeOfService")
_cpt_sm       = _create_semantic_mapping("Normalize CPT4 Codes [OMOP]", _cpt_vocab_uid,       ["Procedure"],  PREPARED_PROCEDURES_UID,  "ProcedureCode")

GENDER_SM_UID     = _gender_sm.uid
RACE_SM_UID       = _race_sm.uid
ETHNICITY_SM_UID  = _ethnicity_sm.uid
VISIT_TYPE_SM_UID = _visit_sm.uid
CPT_SM_UID        = _cpt_sm.uid

# Also export vocabulary UIDs — needed by Step 4 to populate GlobalConfiguration.vocabulary_mapping
GENDER_VOCAB_UID     = _gender_vocab_uid
RACE_VOCAB_UID       = _race_vocab_uid
ETHNICITY_VOCAB_UID  = _ethnicity_vocab_uid
VISIT_TYPE_VOCAB_UID = _visit_vocab_uid
CPT_VOCAB_UID        = _cpt_vocab_uid

# ── 5. Wait for AI term-matching to complete ──────────────────────────────────
print("\nWaiting for AI term-matching (this may take several minutes per mapping)...")
for uid, label in [
    (GENDER_SM_UID,     "Gender"),
    (RACE_SM_UID,       "Race"),
    (ETHNICITY_SM_UID,  "Ethnicity"),
    (VISIT_TYPE_SM_UID, "Visit Type"),
    (CPT_SM_UID,        "CPT4"),
]:
    current = session.semantic_mapping.get_semantic_mapping(uid)
    if current._finished_processing:
        print(f"  {label}: {current.processing_status.value} (already done)")
    else:
        result = current.wait_for_completion(timeout_seconds=900, print_progress=False)
        print(f"  {label}: {result.processing_status.value}")

print("\nAll semantic mappings ready for review — continue to Step 4.")

## Step 4 — Review and Approve Semantic Mappings

The platform has proposed AI-generated term matches for each vocabulary.
**This step requires human review.** Incorrect term matches will silently
produce wrong concept IDs in the harmonized output.

Reference concept IDs (from the Tutorial 4 README):
Gender: Male→8507, Female→8532, Other→8521
Race: Asian→8515, Black→8516, White→8527, Other→8522
Ethnicity: Hispanic→38003563, Non-Hispanic→38003564
Visit: Outpatient→9202, Inpatient→9201, Emergency→9203
CPT: 45378→4287782, 44950→4196867, 99203→4098498, 99212→4098460, 99213→4098462, 99285→4129922

FCP UI equivalent: Dashboard → Harmonization → click a semantic mapping → review/approve terms

In [ ]:
from rhino_health.lib.endpoints.semantic_mapping.semantic_mapping_dataclass import (
    SemanticMappingApproveList, SemanticMappingApproveEntry,
)

def print_and_approve(session, sm_uid, label, approvals):
    """Display AI-proposed matches then submit approvals."""
    try:
        raw = session.semantic_mapping.get_semantic_mapping_data(sm_uid)
        print(f"\n{label} — AI-proposed matches:")
        print(f"  {'Source':<20} → {'Proposed Match':<40} Concept ID")
        print("  " + "─" * 80)
        # raw is a raw API response; iterate results if available
        items = getattr(raw, "results", None) or []
        for t in items:
            print(f"  {getattr(t, 'source_term_name', '?'):<20} → {getattr(t, 'target_term_name', '?'):<40} {getattr(t, 'target_identifier', '?')}")
    except Exception as e:
        print(f"  WARNING: Could not retrieve proposed terms for {label}: {e}")

    try:
        session.semantic_mapping.approve_mappings(
            semantic_mapping_uid=sm_uid,
            mapping_data=SemanticMappingApproveList(
                entries=[SemanticMappingApproveEntry(**entry) for entry in approvals]
            ),
        )
        print(f"  Approved {len(approvals)} term(s) for {label}.")
    except Exception as e:
        print(f"  ERROR approving {label}: {e}")
        raise

In [ ]:
# Gender mapping
print_and_approve(session, GENDER_SM_UID, "Gender → OMOP", [
    {"source_term_name": "Male",   "target_term_name": "Male",   "target_identifier": "8507",     "is_approved": True},
    {"source_term_name": "Female", "target_term_name": "Female", "target_identifier": "8532",     "is_approved": True},
    {"source_term_name": "Other",  "target_term_name": "Other",  "target_identifier": "8521",     "is_approved": True},
])

# Race mapping
print_and_approve(session, RACE_SM_UID, "Race → OMOP", [
    {"source_term_name": "Asian",  "target_term_name": "Asian",                     "target_identifier": "8515",  "is_approved": True},
    {"source_term_name": "Black",  "target_term_name": "Black or African American", "target_identifier": "8516",  "is_approved": True},
    {"source_term_name": "White",  "target_term_name": "White",                     "target_identifier": "8527",  "is_approved": True},
    {"source_term_name": "Other",  "target_term_name": "Other Race",                "target_identifier": "8522",  "is_approved": True},
])

# Ethnicity mapping
print_and_approve(session, ETHNICITY_SM_UID, "Ethnicity → OMOP", [
    {"source_term_name": "Hispanic",     "target_term_name": "Hispanic or Latino",     "target_identifier": "38003563", "is_approved": True},
    {"source_term_name": "Non-Hispanic", "target_term_name": "Not Hispanic or Latino", "target_identifier": "38003564", "is_approved": True},
])

# Visit type mapping
print_and_approve(session, VISIT_TYPE_SM_UID, "Visit Type → OMOP", [
    {"source_term_name": "Outpatient", "target_term_name": "Outpatient Visit",     "target_identifier": "9202", "is_approved": True},
    {"source_term_name": "Inpatient",  "target_term_name": "Inpatient Visit",      "target_identifier": "9201", "is_approved": True},
    {"source_term_name": "Emergency",  "target_term_name": "Emergency Room Visit", "target_identifier": "9203", "is_approved": True},
])

# CPT → OMOP procedure mapping
# The platform creates two sets of entries for CPT:
#   - Float-format ("99285.0") from scanning the ProcedureCode column (stored as float by pandas)
#   - Int-format  ("99285")   from the source vocabulary terms
# We approve BOTH so the overall mapping status reaches "Approved" with no unapproved entries.
# At harmonization time, _normalize_cpt converts 99285.0 → "99285" before the semantic lookup,
# so the int-format approved entries are what actually resolve to OMOP concept IDs.
_cpt_terms = [
    ("45378", "Colonoscopy",                                    "4287782"),
    ("44950", "Appendectomy",                                   "4196867"),
    ("99203", "Office or other outpatient visit, new patient",  "4098498"),
    ("99212", "Office or other outpatient visit, established",  "4098460"),
    ("99213", "Office or other outpatient visit, established",  "4098462"),
    ("99285", "Emergency department visit",                     "4129922"),
]
print_and_approve(session, CPT_SM_UID, "CPT Codes → OMOP", [
    # float format — from column scan (ProcedureCode stored as float in prepared dataset)
    *[{"source_term_name": f"{code}.0", "target_term_name": label, "target_identifier": cid, "is_approved": True}
      for code, label, cid in _cpt_terms],
    # int format — from source vocabulary terms; matched at harmonization time after _normalize_cpt
    *[{"source_term_name": code, "target_term_name": label, "target_identifier": cid, "is_approved": True}
      for code, label, cid in _cpt_terms],
])

## Step 5 — Auto-generate Syntactic Mappings

With all semantic mappings approved, auto-populate the field-level transformation
rules for each syntactic mapping. This creates rules for every source→target
column pair based on the approved semantic mappings and schema alignment.
You can review and manually adjust these rules in the FCP Dashboard if needed.

In [ ]:
from rhino_health.lib.endpoints.syntactic_mapping.syntactic_mapping_dataclass import (
    SyntacticMappingConfig, GlobalConfiguration, TableConfiguration,
    FieldConfiguration, SourceDataField, VocabularyMapping,
    SourceValueTransformation, SecureUUIDTransformation,
    SpecificValueTransformation, SemanticMappingTransformation, SemanticMappingTransformationOutput,
    RowPythonCodeTransformation, RowPythonCodeTransformationOnError,
)
from rhino_health.lib.endpoints.semantic_mapping.semantic_mapping_dataclass import VocabularyType
import json as _json

print("Updating syntactic mapping with OMOP field configurations...")

# RowPythonCodeTransformation exec namespace:
#   - input variable: `row`  (the scalar value from the previous step)
#   - output variable: `output`  (what the executor reads back — NOT `result`)
#   - also available: np, pd, arrow

# Integer ID: hash source value to a consistent 31-bit integer for OMOP ID columns.
_INT_ID_CODE = (
    "import hashlib; "
    "output = abs(int(hashlib.sha256(str(row).encode()).hexdigest(), 16)) % (2**31)"
)
_int_id = lambda: [
    SourceValueTransformation(),
    RowPythonCodeTransformation(code=_INT_ID_CODE, on_error=RowPythonCodeTransformationOnError.FAIL),
]

# Case normalization for Gender and Visit Type.
# Source data has mixed-case variants: MALE/male → Male, EMERGENCY/outpatient → Emergency.
_normalize = RowPythonCodeTransformation(
    code="output = str(row).capitalize()",
    on_error=RowPythonCodeTransformationOnError.FAIL,
)

# CPT code normalization: pandas reads numeric codes as floats (99285.0).
# Convert to clean integer string ("99285") so the vocabulary lookup matches.
_normalize_cpt = RowPythonCodeTransformation(
    code="output = str(int(float(row)))",
    on_error=RowPythonCodeTransformationOnError.FAIL,
)

updated_config = SyntacticMappingConfig(
    global_configuration=GlobalConfiguration(
        data_sources_to_data_schemas=[
            {"patients":   PREPARED_PATIENTS_SCHEMA_UID},
            {"encounters": PREPARED_ENCOUNTERS_SCHEMA_UID},
            {"procedures": PREPARED_PROCEDURES_SCHEMA_UID},
        ],
        vocabulary_mapping=[
            VocabularyMapping(name="gender",     type=VocabularyType.STANDARD, uid=OMOP_VOCABULARY_UID, metadata={"semantic_mapping_uid": GENDER_SM_UID}),
            VocabularyMapping(name="race",        type=VocabularyType.STANDARD, uid=OMOP_VOCABULARY_UID, metadata={"semantic_mapping_uid": RACE_SM_UID}),
            VocabularyMapping(name="ethnicity",   type=VocabularyType.STANDARD, uid=OMOP_VOCABULARY_UID, metadata={"semantic_mapping_uid": ETHNICITY_SM_UID}),
            VocabularyMapping(name="visit_type",  type=VocabularyType.STANDARD, uid=OMOP_VOCABULARY_UID, metadata={"semantic_mapping_uid": VISIT_TYPE_SM_UID}),
            VocabularyMapping(name="cpt",         type=VocabularyType.STANDARD, uid=OMOP_VOCABULARY_UID, metadata={"semantic_mapping_uid": CPT_SM_UID}),
        ],
        output_table_names_to_data_schemas=[
            {"person":               OMOP_PERSON_SCHEMA_UID},
            {"visit_occurrence":     OMOP_VISIT_OCCURRENCE_SCHEMA_UID},
            {"procedure_occurrence": OMOP_PROCEDURE_OCCURRENCE_SCHEMA_UID},
        ],
    ),
    table_configurations=[
        # ── person (source: patients) ─────────────────────────────────────
        TableConfiguration(
            table_name="person",
            field_configurations=[
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="patients", field="patientID")],
                    target_field="person_id",
                    transformations=_int_id(),
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="patients", field="Gender")],
                    target_field="gender_concept_id",
                    transformations=[
                        SourceValueTransformation(),
                        _normalize,  # MALE/male/FEMALE/female → Male/Female
                        SemanticMappingTransformation(
                            vocabulary_name="gender",
                            output=SemanticMappingTransformationOutput.TARGET_IDENTIFIER,
                        ),
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="patients", field="YearOfBirth")],
                    target_field="year_of_birth",
                    transformations=[SourceValueTransformation()],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="patients", field="Race")],
                    target_field="race_concept_id",
                    transformations=[
                        SourceValueTransformation(),
                        SemanticMappingTransformation(
                            vocabulary_name="race",
                            output=SemanticMappingTransformationOutput.TARGET_IDENTIFIER,
                        ),
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="patients", field="Ethnicity")],
                    target_field="ethnicity_concept_id",
                    transformations=[
                        SourceValueTransformation(),
                        SemanticMappingTransformation(
                            vocabulary_name="ethnicity",
                            output=SemanticMappingTransformationOutput.TARGET_IDENTIFIER,
                        ),
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="patients", field="patientID")],
                    target_field="person_source_value",
                    transformations=[SourceValueTransformation()],
                ),
            ],
        ),
        # ── visit_occurrence (source: encounters) ─────────────────────────
        TableConfiguration(
            table_name="visit_occurrence",
            field_configurations=[
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="encounters", field="visitID")],
                    target_field="visit_occurrence_id",
                    transformations=_int_id(),
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="encounters", field="patientID")],
                    target_field="person_id",
                    transformations=_int_id(),
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="encounters", field="TypeOfService")],
                    target_field="visit_concept_id",
                    transformations=[
                        SourceValueTransformation(),
                        _normalize,  # EMERGENCY/outpatient → Emergency/Outpatient
                        SemanticMappingTransformation(
                            vocabulary_name="visit_type",
                            output=SemanticMappingTransformationOutput.TARGET_IDENTIFIER,
                        ),
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="encounters", field="DateOfService")],
                    target_field="visit_start_date",
                    transformations=[SourceValueTransformation()],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="encounters", field="DateOfService")],
                    target_field="visit_end_date",
                    transformations=[SourceValueTransformation()],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="encounters", field="visitID")],
                    target_field="visit_type_concept_id",
                    transformations=[
                        SourceValueTransformation(),
                        SpecificValueTransformation(value=32817),
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="encounters", field="visitID")],
                    target_field="visit_source_value",
                    transformations=[SourceValueTransformation()],
                ),
            ],
        ),
        # ── procedure_occurrence (source: procedures) ─────────────────────
        TableConfiguration(
            table_name="procedure_occurrence",
            field_configurations=[
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="procedures", field="visitID")],
                    target_field="procedure_occurrence_id",
                    transformations=_int_id(),
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="procedures", field="patientID")],
                    target_field="person_id",
                    transformations=_int_id(),
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="procedures", field="ProcedureCode")],
                    target_field="procedure_concept_id",
                    transformations=[
                        SourceValueTransformation(),
                        _normalize_cpt,  # 99285.0 → "99285"
                        SemanticMappingTransformation(
                            vocabulary_name="cpt",
                            output=SemanticMappingTransformationOutput.TARGET_IDENTIFIER,
                        ),
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="procedures", field="ProcedureDate")],
                    target_field="procedure_date",
                    transformations=[SourceValueTransformation()],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="procedures", field="visitID")],
                    target_field="procedure_type_concept_id",
                    transformations=[
                        SourceValueTransformation(),
                        SpecificValueTransformation(value=32817),
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="procedures", field="ProcedureCode")],
                    target_field="procedure_source_value",
                    transformations=[
                        SourceValueTransformation(),
                        _normalize_cpt,  # 99285.0 → "99285"
                    ],
                ),
                FieldConfiguration(
                    source_fields=[SourceDataField(data_source="procedures", field="visitID")],
                    target_field="visit_occurrence_id",
                    transformations=_int_id(),
                ),
            ],
        ),
    ],
)

# Bypass partial_update's exclude_unset=True (strips transformation_type discriminators).
# model_dump_json serializes Enum → string value; json.loads gives a plain dict.
config_payload = _json.loads(updated_config.model_dump_json(by_alias=True))
session.patch(
    f"/syntactic_mappings/{SYNTACTIC_MAPPING_UID}",
    data={"mapping_config": config_payload},
)
print("  Done — syntactic mapping updated with field configurations.")

## Step 6 — Run Harmonization

In [ ]:
from rhino_health.lib.endpoints.syntactic_mapping.syntactic_mapping_dataclass import DataHarmonizationRunInput

print("Starting OMOP harmonization run (all three tables)...")
async_response = session.syntactic_mapping.run_data_harmonization(
    SYNTACTIC_MAPPING_UID,
    DataHarmonizationRunInput(
        input_dataset_uids=[PREPARED_PATIENTS_UID, PREPARED_ENCOUNTERS_UID, PREPARED_PROCEDURES_UID],
        semantic_mapping_uids_by_vocabularies={
            "gender":     GENDER_SM_UID,
            "race":       RACE_SM_UID,
            "ethnicity":  ETHNICITY_SM_UID,
            "visit_type": VISIT_TYPE_SM_UID,
            "cpt":        CPT_SM_UID,
        },
        timeout_seconds=900,
    )
)
code_run_uid = async_response.code_run_uid
print(f"  Run started (code_run_uid={code_run_uid})")

code_run = session.code_run.get_code_run(code_run_uid)
harmonization_result = code_run.wait_for_completion(timeout_seconds=900, print_progress=True)
print(f"\nHarmonization complete: {harmonization_result.status.value}")

# Output datasets ordered by table_configurations: person → visit_occurrence → procedure_occurrence
OMOP_PERSON_UID    = harmonization_result.output_dataset_uids.root[0].root[0].root[0]
OMOP_VISIT_UID     = harmonization_result.output_dataset_uids.root[0].root[1].root[0]
OMOP_PROCEDURE_UID = harmonization_result.output_dataset_uids.root[0].root[2].root[0]

print(f"\n  OMOP Person UID:               {OMOP_PERSON_UID}")
print(f"  OMOP Visit Occurrence UID:     {OMOP_VISIT_UID}")
print(f"  OMOP Procedure Occurrence UID: {OMOP_PROCEDURE_UID}")

In [ ]:
# Run completed in the cell above.
# OMOP_PERSON_UID, OMOP_VISIT_UID, OMOP_PROCEDURE_UID are now set.
print(f"OMOP Person:    {OMOP_PERSON_UID}")
print(f"OMOP Visit:     {OMOP_VISIT_UID}")
print(f"OMOP Procedure: {OMOP_PROCEDURE_UID}")


## Step 7 — Verify Output Datasets

In [ ]:
from rhino_health.lib.metrics import Count, Mean

# Row counts — Count requires a variable (any column will do)
table_id_cols = {
    OMOP_PERSON_UID:    "person_id",
    OMOP_VISIT_UID:     "visit_occurrence_id",
    OMOP_PROCEDURE_UID: "procedure_occurrence_id",
}
outputs = [
    (OMOP_PERSON_UID,    "OMOP Person"),
    (OMOP_VISIT_UID,     "OMOP Visit Occurrence"),
    (OMOP_PROCEDURE_UID, "OMOP Procedure Occurrence"),
]

print(f"{'Table':<35} {'Rows':>8}  UID")
print("─" * 80)
for ds_uid, label in outputs:
    try:
        result = session.dataset.get_dataset_metric(ds_uid, Count(variable=table_id_cols[ds_uid]))
        count = result.output.get("count", result.output)
        print(f"{label:<35} {count:>8,}  {ds_uid}")
    except Exception as e:
        print(f"{label:<35} ERROR: {e}")

# Spot-check: mean of concept ID columns — non-zero confirms semantic mapping produced OMOP IDs
print("\nSpot check — mean of key OMOP concept ID columns:")
checks = [
    (OMOP_PERSON_UID,    "gender_concept_id",   "Person"),
    (OMOP_PERSON_UID,    "race_concept_id",      "Person"),
    (OMOP_PERSON_UID,    "ethnicity_concept_id", "Person"),
    (OMOP_VISIT_UID,     "visit_concept_id",     "Visit"),
    (OMOP_PROCEDURE_UID, "procedure_concept_id", "Procedure"),
]
print(f"  {'Table':<12} {'Column':<25} {'Mean concept_id':>18}")
print("  " + "─" * 58)
for ds_uid, col, tbl in checks:
    try:
        result = session.dataset.get_dataset_metric(ds_uid, Mean(variable=col))
        mean_val = result.output.get("mean", result.output)
        flag = "  ← check mapping" if mean_val is None or mean_val == 0 else ""
        print(f"  {tbl:<12} {col:<25} {mean_val:>18.2f}{flag}")
    except Exception as e:
        print(f"  {tbl:<12} {col:<25} ERROR: {e}")

## FCP UI — What to Check After Running This Notebook

1. **Harmonization Mappings** → Dashboard → Select Your Project → Data Mappings
   - Under "Syntactic Mappings" - you should see 1 Syntactic Mapping with a Target Data Model of OMOP, 3 source schemas, and 3 target tables
      - Click on the mapping: all required target fields should be linked to a source field
   - Under "Semantic Mappings" - you should see 5 Semantic Mappings
      - Each should have a Status of "Approved", with 100% Mapping Coverage
      - Click on any mapping to see a list of SOURCE VALUE(s) to SELECTED TARGET VALUE(s), with approval state and confidence level
   - Under "Custom Vocabularies" - you should see 5 Custom Vocabularies
      - Click on any mapping to see a list of CODE(s) and corresponding DISPLAY NAME(s)

2. **Harmonization Runs** → Dashboard → Select Your Project → Code Runs
   - At least 1 harmonization run (Type DH) appears under the relevant Code Object
   - Click on run → Logs tab → check for warnings about unmapped values

3. **Output Datasets** → Dashboard → Select Your Project → Datasets
   - Three new OMOP datasets appear (person, visit_occurrence, procedure_occurrence)
   - Click any → Analytics tab → inspect distributions
   - A non-zero concept_id=0 rate means some source values were not covered
     by your semantic mapping — return to Step 4 to add missing terms

## Summary — Copy These UIDs

In [ ]:
print("=" * 60)
print("  Tutorial 4 Complete — save these UIDs")
print("=" * 60)
print(f"OMOP_PERSON_UID    = '{OMOP_PERSON_UID}'")
print(f"OMOP_VISIT_UID     = '{OMOP_VISIT_UID}'")
print(f"OMOP_PROCEDURE_UID = '{OMOP_PROCEDURE_UID}'")
print("=" * 60)
print("\nContinue to: Tutorial 5 - Validation")